# Notebook de documentacion, tratamiento datos y entrenamiento


## Equipo
- Alumno 1 : Calabozo, Nicolas Daniel
- Alumno 2 : Lapolla, Martín Facundo

## Librerías

In [ ]:
import ssl
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import cv2
import os
import numpy as np
import uuid

from torch.utils.data import DataLoader
from sklearn.datasets import fetch_lfw_people
from facenet_pytorch import MTCNN, InceptionResnetV1 
from insightface.utils import face_align


import psycopg
from pgvector.psycopg import register_vector 

from src.lib.storage.pgvector_store  import PgVectorEmbeddingStore
from src.lib.schemas import EmbeddingRecord

ssl._create_default_https_context = ssl._create_unverified_context

load_dotenv()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Dispositivo: {device}")

ModuleNotFoundError: No module named 'lib'

In [ ]:
#Setup de BBDD vectorial 
store = PgVectorEmbeddingStore(
    dbname=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("POSTGRES_HOST"),
    port=os.getenv("POSTGRES_PORT", "5432"),
    embedding_dim=os.getenv("EMBEDDING_DIM")
)

### EDA

In [7]:
lfw_people = fetch_lfw_people(
    data_home='./data',           
    min_faces_per_person=20,                    
    color=True, 
    slice_=None,
    resize=1.0,                   
    download_if_missing=True     
)

In [ ]:
def preparar_modelo(num_clases):
    """
    Carga InceptionResnetV1 (FaceNet) y adapta el head para clasificación.
    """
    
    model = InceptionResnetV1(pretrained='vggface2').to(device)
    
    for param in model.parameters():
        param.requires_grad = False

    model.logits = nn.Linear(512, num_clases).to(device)
    
    for param in model.logits.parameters():
        param.requires_grad = True
        
    print(f"Modelo InceptionResnetV1 listo con {num_clases} clases.")
    return model

modelo_entrenado = preparar_modelo()

In [ ]:
def procesar_y_guardar_rostro(path_imagen, etiqueta, store: PgVectorEmbeddingStore):
    img = cv2.imread(path_imagen)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    boxes, probs, landmarks = mtcnn.detect(img_rgb, landmarks=True)
    if boxes is None:
        print("No se detectó ningún rostro.")
        return

    for i, box in enumerate(boxes):
        face_img = face_align.norm_crop(img, landmarks[i]) 
        embedding = modelo_entrenado(transform(face_img).unsqueeze(0))
        embedding_list = embedding.detach().cpu().numpy().flatten().tolist()
        store.append(
            EmbeddingRecord(
                id_imagen=str(uuid.uuid4()),
                embedding=embedding_list,
                path=path_imagen,
                etiqueta=etiqueta,
                metadata={"detector": "MTCNN", "aligner": "InsightFace"}
            )
        )
    
    print(f"Rostro de {etiqueta} procesado y guardado.")

In [ ]:
def entrenar_y_exportar(model, train_loader, epochs=10):
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(model.logits.parameters(), lr=0.001)
    
    model.train()
    for epoch in range(epochs):
        curr_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            curr_loss += loss.item()
            
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {curr_loss/len(train_loader):.4f}")
    
    os.makedirs('models', exist_ok=True)
    torch.save(model.state_dict(), 'models/face_classifier_v1.pth')
